In [ ]:
import os, math

WIN_LENGTH = 5

def make_board(n):
    """Tạo bảng NxN đánh số từ 1..N*N."""
    return list(range(1, n * n + 1))

def get_winner(board, n, k=WIN_LENGTH):
    """Kiểm tra thắng thua trên bảng NxN với chuỗi k liên tiếp."""
    for r in range(n):
        for c in range(n):
            val = board[r * n + c]
            if val not in ("X", "O"):
                continue
            # ngang
            if c + k <= n:
                if all(board[r * n + c + i] == val for i in range(k)):
                    return val
            # dọc
            if r + k <= n:
                if all(board[(r + i) * n + c] == val for i in range(k)):
                    return val
            # chéo xuống phải
            if r + k <= n and c + k <= n:
                if all(board[(r + i) * n + c + i] == val for i in range(k)):
                    return val
            # chéo xuống trái
            if r + k <= n and c - k + 1 >= 0:
                if all(board[(r + i) * n + c - i] == val for i in range(k)):
                    return val
    return None

def get_available(board):
    return [cell for cell in board if cell not in ("X", "O")]

def print_board(board, n):
    os.system('cls' if os.name == 'nt' else 'clear')
    col_w = len(str(n * n)) + 1
    sep = "+" + ("-" * col_w + "+") * n
    print(sep)
    for r in range(n):
        row = "|"
        for c in range(n):
            val = board[r * n + c]
            cell_str = str(val).center(col_w)
            row += cell_str + "|"
        print(row)
        print(sep)

# ──── Heuristic evaluation (dùng cho bảng lớn, tránh minimax sâu vô hạn) ────

def score_line(line, k, ai, human):
    """Tính điểm 1 đường (list các ô)."""
    score = 0
    for start in range(len(line) - k + 1):
        window = line[start:start + k]
        ai_cnt = window.count(ai)
        hu_cnt = window.count(human)
        if hu_cnt == 0 and ai_cnt > 0:
            score += 10 ** ai_cnt
        elif ai_cnt == 0 and hu_cnt > 0:
            score -= 10 ** hu_cnt
    return score

def evaluate(board, n, k, ai, human):
    """Heuristic evaluation của toàn bộ bảng."""
    score = 0
    # hàng
    for r in range(n):
        line = [board[r * n + c] for c in range(n)]
        score += score_line(line, k, ai, human)
    # cột
    for c in range(n):
        line = [board[r * n + c] for r in range(n)]
        score += score_line(line, k, ai, human)
    # chéo chính
    for start_r in range(n - k + 1):
        for start_c in range(n - k + 1):
            line = [board[(start_r + i) * n + start_c + i] for i in range(min(n - start_r, n - start_c))]
            score += score_line(line, k, ai, human)
    # chéo phụ
    for start_r in range(n - k + 1):
        for start_c in range(k - 1, n):
            line = [board[(start_r + i) * n + start_c - i] for i in range(min(n - start_r, start_c + 1))]
            score += score_line(line, k, ai, human)
    return score

def minimax(position, n, k, depth, max_depth, alpha, beta, is_maximizing, ai, human):
    """
    Minimax với alpha-beta pruning và giới hạn độ sâu (dùng heuristic khi hết độ sâu).
    """
    winner = get_winner(position, n, k)
    if winner == ai:
        return 10000 - depth
    if winner == human:
        return -10000 + depth
    available = get_available(position)
    if not available:
        return 0
    if depth >= max_depth:
        return evaluate(position, n, k, ai, human)

    if is_maximizing:
        max_eval = -math.inf
        for cell in available:
            position[cell - 1] = ai
            val = minimax(position, n, k, depth + 1, max_depth, alpha, beta, False, ai, human)
            position[cell - 1] = cell
            max_eval = max(max_eval, val)
            alpha = max(alpha, val)
            if beta <= alpha:
                break
        return max_eval
    else:
        min_eval = math.inf
        for cell in available:
            position[cell - 1] = human
            val = minimax(position, n, k, depth + 1, max_depth, alpha, beta, True, ai, human)
            position[cell - 1] = cell
            min_eval = min(min_eval, val)
            beta = min(beta, val)
            if beta <= alpha:
                break
        return min_eval

def find_best_move(board, n, k, ai, human, max_depth=4):
    """Tìm nước đi tốt nhất cho AI."""
    best_val = -math.inf
    best_move = -1
    available = get_available(board)

    # Ưu tiên: nếu AI thắng ngay -> chọn; nếu chặn human thắng -> chặn
    for cell in available:
        board[cell - 1] = ai
        if get_winner(board, n, k) == ai:
            board[cell - 1] = cell
            return cell
        board[cell - 1] = cell

    for cell in available:
        board[cell - 1] = human
        if get_winner(board, n, k) == human:
            board[cell - 1] = cell
            return cell
        board[cell - 1] = cell

    for cell in available:
        board[cell - 1] = ai
        move_val = minimax(board, n, k, 0, max_depth, -math.inf, math.inf, False, ai, human)
        board[cell - 1] = cell
        if move_val > best_val:
            best_val = move_val
            best_move = cell
    return best_move

def play(n=3, k=None, max_depth=4):
    if k is None:
        k = min(n, WIN_LENGTH)
    board = make_board(n)
    player = input(f"[{n}x{n}, thắng khi có {k} liên tiếp] Chơi X hay O? ").strip().upper()
    while player not in ("X", "O"):
        player = input("Nhập X hoặc O: ").strip().upper()
    ai = "O" if player == "X" else "X"
    turn = "X"
    moves = 0

    while True:
        winner = get_winner(board, n, k)
        if winner:
            print_board(board, n)
            print(f"🏆 {winner} THẮNG!")
            return
        if not get_available(board):
            print_board(board, n)
            print("🤝 HÒA!")
            return

        if turn == ai:
            print_board(board, n)
            print("AI đang suy nghĩ...")
            cell = find_best_move(board, n, k, ai, player, max_depth)
            board[cell - 1] = ai
            print(f"AI chọn ô {cell}")
        else:
            print_board(board, n)
            while True:
                try:
                    inp = int(input(f"Lượt {player}, nhập số ô (1-{n*n}): "))
                    if inp in board and board[inp - 1] not in ("X", "O"):
                        board[inp - 1] = player
                        break
                    else:
                        print("Ô không hợp lệ hoặc đã bị chiếm.")
                except ValueError:
                    print("Vui lòng nhập số.")

        turn = player if turn == ai else ai
        moves += 1

def main():
    print("=== Bài 3: Minimax TicTacToe NxN ===")
    print("Chọn kích thước bảng:")
    print("  1. 3x3  (win=3)")
    print("  2. 5x5  (win=5)")
    print("  3. 10x10 (win=5)")
    print("  4. Tùy chỉnh NxN")
    choice = input("Lựa chọn: ").strip()
    if choice == "1":
        play(3, 3, max_depth=9)
    elif choice == "2":
        play(5, 5, max_depth=3)
    elif choice == "3":
        play(10, 5, max_depth=2)
    elif choice == "4":
        n = int(input("Nhập N: "))
        k = int(input(f"Nhập số ký tự thắng (K≤{n}): "))
        d = int(input("Nhập độ sâu tìm kiếm (max_depth, VD: 3): "))
        play(n, k, d)
    else:
        print("Lựa chọn không hợp lệ.")

if __name__ == "__main__":
    main()

In [ ]:
import os, math

WIN_LENGTH = 5

def make_board(n):
    return list(range(1, n * n + 1))

def get_winner(board, n, k=WIN_LENGTH):
    for r in range(n):
        for c in range(n):
            val = board[r * n + c]
            if val not in ("X", "O"):
                continue
            if c + k <= n and all(board[r * n + c + i] == val for i in range(k)):
                return val
            if r + k <= n and all(board[(r + i) * n + c] == val for i in range(k)):
                return val
            if r + k <= n and c + k <= n and all(board[(r + i) * n + c + i] == val for i in range(k)):
                return val
            if r + k <= n and c - k + 1 >= 0 and all(board[(r + i) * n + c - i] == val for i in range(k)):
                return val
    return None

def get_available(board):
    return [cell for cell in board if cell not in ("X", "O")]

def print_board(board, n):
    os.system('cls' if os.name == 'nt' else 'clear')
    col_w = len(str(n * n)) + 1
    sep = "+" + ("-" * col_w + "+") * n
    print(sep)
    for r in range(n):
        row = "|"
        for c in range(n):
            val = board[r * n + c]
            row += str(val).center(col_w) + "|"
        print(row)
        print(sep)

def score_line(line, k, ai, human):
    score = 0
    for s in range(len(line) - k + 1):
        w = line[s:s + k]
        ai_c = w.count(ai)
        hu_c = w.count(human)
        if hu_c == 0 and ai_c > 0:
            score += 10 ** ai_c
        elif ai_c == 0 and hu_c > 0:
            score -= 10 ** hu_c
    return score

def evaluate(board, n, k, ai, human):
    score = 0
    for r in range(n):
        line = [board[r * n + c] for c in range(n)]
        score += score_line(line, k, ai, human)
    for c in range(n):
        line = [board[r * n + c] for r in range(n)]
        score += score_line(line, k, ai, human)
    for sr in range(n):
        for sc in range(n):
            length = min(n - sr, n - sc)
            if length >= k:
                line = [board[(sr + i) * n + sc + i] for i in range(length)]
                score += score_line(line, k, ai, human)
    for sr in range(n):
        for sc in range(n):
            length = min(n - sr, sc + 1)
            if length >= k:
                line = [board[(sr + i) * n + sc - i] for i in range(length)]
                score += score_line(line, k, ai, human)
    return score


pruned_count = 0
nodes_visited = 0

def alpha_beta(position, n, k, depth, max_depth, alpha, beta, is_maximizing, ai, human):
    """
    Alpha-Beta Pruning thuần túy.
    Trả về (score, pruned_local, nodes_local).
    """
    global pruned_count, nodes_visited
    nodes_visited += 1

    winner = get_winner(position, n, k)
    if winner == ai:
        return 10000 - depth
    if winner == human:
        return -10000 + depth
    available = get_available(position)
    if not available:
        return 0
    if depth >= max_depth:
        return evaluate(position, n, k, ai, human)

    if is_maximizing:
        max_eval = -math.inf
        for cell in available:
            position[cell - 1] = ai
            val = alpha_beta(position, n, k, depth + 1, max_depth, alpha, beta, False, ai, human)
            position[cell - 1] = cell
            max_eval = max(max_eval, val)
            alpha = max(alpha, val)
            if beta <= alpha:
                pruned_count += 1
                break
        return max_eval
    else:
        min_eval = math.inf
        for cell in available:
            position[cell - 1] = human
            val = alpha_beta(position, n, k, depth + 1, max_depth, alpha, beta, True, ai, human)
            position[cell - 1] = cell
            min_eval = min(min_eval, val)
            beta = min(beta, val)
            if beta <= alpha:
                pruned_count += 1
                break
        return min_eval

def find_best_move(board, n, k, ai, human, max_depth=4):
    global pruned_count, nodes_visited
    pruned_count = 0
    nodes_visited = 0

    available = get_available(board)
    # Nước thắng ngay
    for cell in available:
        board[cell - 1] = ai
        if get_winner(board, n, k) == ai:
            board[cell - 1] = cell
            return cell, 0, 0
        board[cell - 1] = cell
    # Chặn đối thủ
    for cell in available:
        board[cell - 1] = human
        if get_winner(board, n, k) == human:
            board[cell - 1] = cell
            return cell, 0, 0
        board[cell - 1] = cell

    best_val = -math.inf
    best_move = -1
    for cell in available:
        board[cell - 1] = ai
        move_val = alpha_beta(board, n, k, 0, max_depth, -math.inf, math.inf, False, ai, human)
        board[cell - 1] = cell
        if move_val > best_val:
            best_val = move_val
            best_move = cell
    return best_move, nodes_visited, pruned_count

def play(n=3, k=None, max_depth=4):
    if k is None:
        k = min(n, WIN_LENGTH)
    board = make_board(n)
    player = input(f"[{n}x{n}, thắng khi có {k} liên tiếp] Chơi X hay O? ").strip().upper()
    while player not in ("X", "O"):
        player = input("Nhập X hoặc O: ").strip().upper()
    ai = "O" if player == "X" else "X"
    turn = "X"

    while True:
        winner = get_winner(board, n, k)
        if winner:
            print_board(board, n)
            print(f"🏆 {winner} THẮNG!")
            return
        if not get_available(board):
            print_board(board, n)
            print("🤝 HÒA!")
            return

        if turn == ai:
            print_board(board, n)
            print("AI đang tính (Alpha-Beta)...")
            cell, nodes, pruned = find_best_move(board, n, k, ai, player, max_depth)
            board[cell - 1] = ai
            print(f"AI chọn ô {cell} | Nodes duyệt: {nodes} | Lần cắt tỉa: {pruned}")
        else:
            print_board(board, n)
            while True:
                try:
                    inp = int(input(f"Lượt {player}, nhập số ô (1-{n*n}): "))
                    if isinstance(board[inp - 1], int):
                        board[inp - 1] = player
                        break
                    else:
                        print("Ô đã bị chiếm.")
                except (ValueError, IndexError):
                    print("Vui lòng nhập số hợp lệ.")

        turn = player if turn == ai else ai

def main():
    print("=== Bài 4: Alpha-Beta Pruning TicTacToe NxN ===")
    print("Chọn kích thước bảng:")
    print("  1. 3x3  (win=3)")
    print("  2. 5x5  (win=5)")
    print("  3. 10x10 (win=5)")
    print("  4. Tùy chỉnh NxN")
    choice = input("Lựa chọn: ").strip()
    if choice == "1":
        play(3, 3, max_depth=9)
    elif choice == "2":
        play(5, 5, max_depth=3)
    elif choice == "3":
        play(10, 5, max_depth=2)
    elif choice == "4":
        n = int(input("Nhập N: "))
        k = int(input(f"Nhập số ký tự thắng (K≤{n}): "))
        d = int(input("Nhập max_depth (VD: 3): "))
        play(n, k, d)
    else:
        print("Lựa chọn không hợp lệ.")

if __name__ == "__main__":
    main()

In [ ]:
import os, math, copy


CỜTƯỚNG_INIT = [
    ['r','n','e','a','k','a','e','n','r'],
    ['.','.','.','.','.','.','.','.','.'],
    ['.','c','.','.','.','.','.','c','.'],
    ['p','.','p','.','p','.','p','.','p'],
    ['.','.','.','.','.','.','.','.','.'],
    ['.','.','.','.','.','.','.','.','.'],
    ['P','.','P','.','P','.','P','.','P'],
    ['.','C','.','.','.','.','.','C','.'],
    ['.','.','.','.','.','.','.','.','.'],
    ['R','N','E','A','K','A','E','N','R'],
]

# Điểm giá trị quân
PIECE_VALUE_CT = {'K':10000,'A':120,'E':120,'R':600,'C':300,'N':300,'P':60,
                  'k':-10000,'a':-120,'e':-120,'r':-600,'c':-300,'n':-300,'p':-60}

def ct_print(board):
    os.system('cls' if os.name == 'nt' else 'clear')
    print("    " + "  ".join(str(i) for i in range(9)))
    print("  +" + "---" * 9 + "-+")
    for r, row in enumerate(board):
        print(f"{r:2d}| " + "  ".join(row) + " |")
    print("  +" + "---" * 9 + "-+")
    print("  Đỏ=Hoa, Đen=thường | K=Tướng R=Xe N=Mã C=Pháo A=Sĩ E=Tượng P=Tốt")

def ct_evaluate(board):
    score = 0
    for row in board:
        for cell in row:
            score += PIECE_VALUE_CT.get(cell, 0)
    return score

def ct_in_bounds(r, c):
    return 0 <= r < 10 and 0 <= c < 9

def ct_is_red(p): return p.isupper() and p != '.'
def ct_is_black(p): return p.islower() and p != '.'

def ct_moves(board, is_red):
    """Sinh tất cả nước đi hợp lệ (đơn giản hóa cho mục đích học thuật)."""
    moves = []
    for r in range(10):
        for c in range(9):
            p = board[r][c]
            if p == '.':
                continue
            if is_red and not ct_is_red(p):
                continue
            if not is_red and not ct_is_black(p):
                continue
            pt = p.upper()
            # Xe (Rook)
            if pt == 'R':
                for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                    nr, nc = r+dr, c+dc
                    while ct_in_bounds(nr, nc):
                        dest = board[nr][nc]
                        if dest == '.':
                            moves.append(((r,c),(nr,nc)))
                        elif (is_red and ct_is_black(dest)) or (not is_red and ct_is_red(dest)):
                            moves.append(((r,c),(nr,nc)))
                            break
                        else:
                            break
                        nr += dr; nc += dc
            # Pháo (Cannon) – nhảy qua 1 quân khi ăn
            elif pt == 'C':
                for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                    nr, nc = r+dr, c+dc
                    screen = False
                    while ct_in_bounds(nr, nc):
                        dest = board[nr][nc]
                        if not screen:
                            if dest == '.':
                                moves.append(((r,c),(nr,nc)))
                            else:
                                screen = True
                        else:
                            if dest != '.':
                                if (is_red and ct_is_black(dest)) or (not is_red and ct_is_red(dest)):
                                    moves.append(((r,c),(nr,nc)))
                                break
                        nr += dr; nc += dc
            # Mã (Knight)
            elif pt == 'N':
                for dr, dc, br, bc in [(-2,1,-1,0),(-2,-1,-1,0),(2,1,1,0),(2,-1,1,0),
                                        (-1,2,0,1),(-1,-2,0,-1),(1,2,0,1),(1,-2,0,-1)]:
                    if board[r+br][c+bc] == '.' if ct_in_bounds(r+br,c+bc) else False:
                        nr, nc = r+dr, c+dc
                        if ct_in_bounds(nr,nc):
                            dest = board[nr][nc]
                            if dest == '.' or (is_red and ct_is_black(dest)) or (not is_red and ct_is_red(dest)):
                                moves.append(((r,c),(nr,nc)))
            # Tốt (Pawn)
            elif pt == 'P':
                if is_red:
                    dirs = [(-1,0)] + ([(0,-1),(0,1)] if r <= 4 else [])
                else:
                    dirs = [(1,0)] + ([(0,-1),(0,1)] if r >= 5 else [])
                for dr, dc in dirs:
                    nr, nc = r+dr, c+dc
                    if ct_in_bounds(nr,nc):
                        dest = board[nr][nc]
                        if dest == '.' or (is_red and ct_is_black(dest)) or (not is_red and ct_is_red(dest)):
                            moves.append(((r,c),(nr,nc)))
            # Tướng và Sĩ (đơn giản hóa)
            elif pt in ('K','A'):
                for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                    nr, nc = r+dr, c+dc
                    if is_red and not (7<=nr<=9 and 3<=nc<=5): continue
                    if not is_red and not (0<=nr<=2 and 3<=nc<=5): continue
                    if ct_in_bounds(nr,nc):
                        dest = board[nr][nc]
                        if dest == '.' or (is_red and ct_is_black(dest)) or (not is_red and ct_is_red(dest)):
                            moves.append(((r,c),(nr,nc)))
    return moves

def ct_apply_move(board, move):
    b = copy.deepcopy(board)
    (r1,c1),(r2,c2) = move
    b[r2][c2] = b[r1][c1]
    b[r1][c1] = '.'
    return b

def ct_alpha_beta(board, depth, alpha, beta, is_red):
    if depth == 0:
        return ct_evaluate(board), None
    moves = ct_moves(board, is_red)
    if not moves:
        return ct_evaluate(board), None
    best_move = None
    if is_red:
        max_eval = -math.inf
        for m in moves:
            nb = ct_apply_move(board, m)
            val, _ = ct_alpha_beta(nb, depth-1, alpha, beta, False)
            if val > max_eval:
                max_eval = val; best_move = m
            alpha = max(alpha, val)
            if beta <= alpha: break
        return max_eval, best_move
    else:
        min_eval = math.inf
        for m in moves:
            nb = ct_apply_move(board, m)
            val, _ = ct_alpha_beta(nb, depth-1, alpha, beta, True)
            if val < min_eval:
                min_eval = val; best_move = m
            beta = min(beta, val)
            if beta <= alpha: break
        return min_eval, best_move

def play_co_tuong(depth=2):
    board = copy.deepcopy(CỜTƯỚNG_INIT)
    color = input("Chơi Đen (b) hay Đỏ (r)? ").strip().lower()
    human_red = (color == 'r')
    turn_red = True  # Đỏ đi trước

    while True:
        ct_print(board)
        moves = ct_moves(board, turn_red)
        if not moves:
            print("Hết nước – thua!")
            break
        is_human = (turn_red == human_red)
        if is_human:
            print("Nước hợp lệ (VD: 9 4 8 4  = từ (9,4) đến (8,4))")
            while True:
                try:
                    r1,c1,r2,c2 = map(int, input("Nhập nước: ").split())
                    m = ((r1,c1),(r2,c2))
                    if m in moves:
                        board = ct_apply_move(board, m); break
                    else: print("Nước không hợp lệ.")
                except: print("Nhập lại.")
        else:
            print("AI đang tính...")
            _, m = ct_alpha_beta(board, depth, -math.inf, math.inf, turn_red)
            if m:
                board = ct_apply_move(board, m)
                print(f"AI đi: {m[0]} -> {m[1]}")
        turn_red = not turn_red


CHESS_INIT = [
    ['r','n','b','q','k','b','n','r'],
    ['p','p','p','p','p','p','p','p'],
    ['.','.','.','.','.','.','.','.'],
    ['.','.','.','.','.','.','.','.'],
    ['.','.','.','.','.','.','.','.'],
    ['.','.','.','.','.','.','.','.'],
    ['P','P','P','P','P','P','P','P'],
    ['R','N','B','Q','K','B','N','R'],
]

PIECE_VALUE_CHESS = {'K':20000,'Q':900,'R':500,'B':330,'N':320,'P':100,
                     'k':-20000,'q':-900,'r':-500,'b':-330,'n':-320,'p':-100}

def chess_print(board):
    os.system('cls' if os.name == 'nt' else 'clear')
    print("    a  b  c  d  e  f  g  h")
    print("  +" + "--" * 8 + "+")
    for i, row in enumerate(board):
        print(f"{8-i:2d}| " + "  ".join(row) + " |")
    print("  +" + "--" * 8 + "+")

def chess_evaluate(board):
    score = 0
    for row in board:
        for c in row:
            score += PIECE_VALUE_CHESS.get(c, 0)
    return score

def chess_is_white(p): return p.isupper() and p != '.'
def chess_is_black(p): return p.islower() and p != '.'

def chess_moves(board, white_to_move):
    """Sinh nước đi cờ vua (đơn giản hóa – không en passant, castling)."""
    moves = []
    for r in range(8):
        for c in range(8):
            p = board[r][c]
            if p == '.': continue
            if white_to_move and not chess_is_white(p): continue
            if not white_to_move and not chess_is_black(p): continue
            pt = p.upper()
            def can_go(nr, nc):
                if not (0<=nr<8 and 0<=nc<8): return False
                dest = board[nr][nc]
                if white_to_move: return not chess_is_white(dest)
                return not chess_is_black(dest)
            def slide(drs):
                for dr, dc in drs:
                    nr,nc = r+dr,c+dc
                    while 0<=nr<8 and 0<=nc<8:
                        dest=board[nr][nc]
                        if (white_to_move and chess_is_white(dest)) or (not white_to_move and chess_is_black(dest)):
                            break
                        moves.append(((r,c),(nr,nc)))
                        if dest != '.': break
                        nr+=dr; nc+=dc
            if pt == 'P':
                d = -1 if white_to_move else 1
                start_r = 6 if white_to_move else 1
                if 0<=r+d<8 and board[r+d][c] == '.':
                    moves.append(((r,c),(r+d,c)))
                    if r == start_r and board[r+2*d][c] == '.':
                        moves.append(((r,c),(r+2*d,c)))
                for dc in [-1,1]:
                    nr,nc=r+d,c+dc
                    if 0<=nr<8 and 0<=nc<8:
                        dest=board[nr][nc]
                        if (white_to_move and chess_is_black(dest)) or (not white_to_move and chess_is_white(dest)):
                            moves.append(((r,c),(nr,nc)))
            elif pt in ('R','Q'):
                slide([(-1,0),(1,0),(0,-1),(0,1)])
            if pt in ('B','Q'):
                slide([(-1,-1),(-1,1),(1,-1),(1,1)])
            elif pt == 'N':
                for dr,dc in [(-2,-1),(-2,1),(2,-1),(2,1),(-1,-2),(-1,2),(1,-2),(1,2)]:
                    if can_go(r+dr,c+dc): moves.append(((r,c),(r+dr,c+dc)))
            elif pt == 'K':
                for dr in [-1,0,1]:
                    for dc in [-1,0,1]:
                        if dr==0 and dc==0: continue
                        if can_go(r+dr,c+dc): moves.append(((r,c),(r+dr,c+dc)))
    return moves

def chess_apply(board, move):
    b = copy.deepcopy(board)
    (r1,c1),(r2,c2) = move
    p = b[r1][c1]
    b[r2][c2] = p; b[r1][c1] = '.'
    # phong tốt
    if p == 'P' and r2 == 0: b[r2][c2] = 'Q'
    if p == 'p' and r2 == 7: b[r2][c2] = 'q'
    return b

def chess_alpha_beta(board, depth, alpha, beta, white):
    if depth == 0: return chess_evaluate(board), None
    moves = chess_moves(board, white)
    if not moves: return chess_evaluate(board), None
    best = None
    if white:
        best_v = -math.inf
        for m in moves:
            nb = chess_apply(board, m)
            v, _ = chess_alpha_beta(nb, depth-1, alpha, beta, False)
            if v > best_v: best_v=v; best=m
            alpha = max(alpha,v)
            if beta<=alpha: break
        return best_v, best
    else:
        best_v = math.inf
        for m in moves:
            nb = chess_apply(board, m)
            v, _ = chess_alpha_beta(nb, depth-1, alpha, beta, True)
            if v < best_v: best_v=v; best=m
            beta = min(beta,v)
            if beta<=alpha: break
        return best_v, best

def play_chess(depth=3):
    board = copy.deepcopy(CHESS_INIT)
    color = input("Chơi Trắng (w) hay Đen (b)? ").strip().lower()
    human_white = (color == 'w')
    white_turn = True

    while True:
        chess_print(board)
        moves = chess_moves(board, white_turn)
        if not moves:
            print("Hết nước!")
            break
        is_human = (white_turn == human_white)
        if is_human:
            print("Nhập nước (VD: e2 e4)")
            col_map = {'a':0,'b':1,'c':2,'d':3,'e':4,'f':5,'g':6,'h':7}
            while True:
                try:
                    src, dst = input("Nước đi: ").split()
                    r1=8-int(src[1]); c1=col_map[src[0]]
                    r2=8-int(dst[1]); c2=col_map[dst[0]]
                    m = ((r1,c1),(r2,c2))
                    if m in moves:
                        board = chess_apply(board, m); break
                    else: print("Nước không hợp lệ.")
                except: print("Nhập lại VD: e2 e4")
        else:
            print("AI đang tính...")
            _, m = chess_alpha_beta(board, depth, -math.inf, math.inf, white_turn)
            if m:
                board = chess_apply(board, m)
                col_rev = {0:'a',1:'b',2:'c',3:'d',4:'e',5:'f',6:'g',7:'h'}
                (r1,c1),(r2,c2) = m
                print(f"AI đi: {col_rev[c1]}{8-r1} -> {col_rev[c2]}{8-r2}")
        white_turn = not white_turn


def go_print(board, n):
    os.system('cls' if os.name == 'nt' else 'clear')
    syms = {0:'.', 1:'X', -1:'O'}
    print("   " + " ".join(f"{c:2d}" for c in range(n)))
    for r in range(n):
        print(f"{r:2d} " + "  ".join(syms[board[r][c]] for c in range(n)))

def go_liberties(board, r, c, n, visited=None):
    """Đếm khí của nhóm quân tại (r,c)."""
    if visited is None: visited = set()
    color = board[r][c]
    if color == 0: return 0
    visited.add((r,c))
    libs = 0
    for dr,dc in [(-1,0),(1,0),(0,-1),(0,1)]:
        nr,nc = r+dr, c+dc
        if 0<=nr<n and 0<=nc<n:
            if board[nr][nc] == 0: libs += 1
            elif board[nr][nc] == color and (nr,nc) not in visited:
                libs += go_liberties(board, nr, nc, n, visited)
    return libs

def go_evaluate(board, n):
    """Heuristic: Territory + số quân."""
    score = 0
    for r in range(n):
        for c in range(n):
            if board[r][c] == 1: score += 10
            elif board[r][c] == -1: score -= 10
    return score

def go_moves(board, n, color):
    """Sinh nước đi hợp lệ (đơn giản – không ko rule)."""
    moves = []
    for r in range(n):
        for c in range(n):
            if board[r][c] == 0:
                moves.append((r,c))
    return moves

def go_apply(board, r, c, color, n):
    b = copy.deepcopy(board)
    b[r][c] = color
    # Bắt quân đối thủ hết khí
    opp = -color
    for dr,dc in [(-1,0),(1,0),(0,-1),(0,1)]:
        nr,nc = r+dr, c+dc
        if 0<=nr<n and 0<=nc<n and b[nr][nc] == opp:
            if go_liberties(b, nr, nc, n) == 0:
                # xóa nhóm
                def remove(board, sr, sc):
                    if board[sr][sc] != opp: return
                    board[sr][sc] = 0
                    for ddr,ddc in [(-1,0),(1,0),(0,-1),(0,1)]:
                        nnr,nnc = sr+ddr, sc+ddc
                        if 0<=nnr<n and 0<=nnc<n:
                            remove(board, nnr, nnc)
                remove(b, nr, nc)
    return b

def go_alpha_beta(board, n, depth, alpha, beta, is_max, color=1):
    if depth == 0: return go_evaluate(board, n), None
    moves = go_moves(board, n, color)
    moves = moves[:20]  # giới hạn nhánh cho bảng lớn
    if not moves: return go_evaluate(board, n), None
    best = None
    opp = -color
    if is_max:
        best_v = -math.inf
        for r,c in moves:
            nb = go_apply(board, r, c, color, n)
            v, _ = go_alpha_beta(nb, n, depth-1, alpha, beta, False, opp)
            if v > best_v: best_v=v; best=(r,c)
            alpha = max(alpha,v)
            if beta<=alpha: break
        return best_v, best
    else:
        best_v = math.inf
        for r,c in moves:
            nb = go_apply(board, r, c, color, n)
            v, _ = go_alpha_beta(nb, n, depth-1, alpha, beta, True, opp)
            if v < best_v: best_v=v; best=(r,c)
            beta = min(beta,v)
            if beta<=alpha: break
        return best_v, best

def play_go(n=9, depth=2):
    board = [[0]*n for _ in range(n)]
    color_in = input(f"[{n}x{n} Cờ Vây] Chơi X(1) hay O(-1)? ").strip().upper()
    human = 1 if color_in == 'X' else -1
    ai = -human
    turn = 1  # X đi trước
    pass_count = 0

    while pass_count < 2:
        go_print(board, n)
        is_human = (turn == human)
        if is_human:
            inp = input(f"Lượt {'X' if turn==1 else 'O'}, nhập 'r c' hoặc 'pass': ").strip()
            if inp == 'pass':
                pass_count += 1
            else:
                try:
                    r,c = map(int, inp.split())
                    if board[r][c] == 0:
                        board = go_apply(board, r, c, turn, n)
                        pass_count = 0
                    else: print("Ô đã có quân."); continue
                except: print("Nhập lại."); continue
        else:
            print("AI đang tính...")
            _, m = go_alpha_beta(board, n, depth, -math.inf, math.inf, True, ai)
            if m:
                board = go_apply(board, m[0], m[1], ai, n)
                print(f"AI đặt tại ({m[0]}, {m[1]})")
                pass_count = 0
            else:
                print("AI pass.")
                pass_count += 1
        turn = -turn

    go_print(board, n)
    score = go_evaluate(board, n)
    print(f"Tổng điểm: {score} ({'X thắng' if score>0 else 'O thắng' if score<0 else 'Hòa'})")


def main():
    print("=== Bài 5: Alpha-Beta cho Cờ Tướng / Cờ Vua / Cờ Vây ===")
    print("  1. Cờ Tướng (9x10)")
    print("  2. Cờ Vua   (8x8)")
    print("  3. Cờ Vây   (9x9)")
    c = input("Chọn: ").strip()
    if c == '1':
        d = int(input("Độ sâu AI (2-3 khuyến nghị): ") or 2)
        play_co_tuong(d)
    elif c == '2':
        d = int(input("Độ sâu AI (2-3 khuyến nghị): ") or 3)
        play_chess(d)
    elif c == '3':
        n = int(input("Kích thước bảng (9/13/19, mặc định 9): ") or 9)
        d = int(input("Độ sâu AI (1-2 khuyến nghị): ") or 2)
        play_go(n, d)
    else:
        print("Không hợp lệ.")

if __name__ == "__main__":
    main()